<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [3]:
mlflow.set_experiment(
    "assignment"
)

<Experiment: artifact_location=('file:c:/Users/PC/OneDrive/Área de '
 'Trabalho/aaaaaaa/atividade-04-deep-learning-i-deadcube04/notebooks/mlruns/1'), creation_time=1779423544802, experiment_id='1', last_update_time=1779423544802, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Resposta:**

- Formato original das imagens: `(32, 32, 3)`.
- Após o flatten, cada imagem passa a ter `3072` features, pois `32 × 32 × 3 = 3072`.
- O flatten é necessário porque a MLP trabalha com vetores 2D, não com tensores 3D de imagem.
- A normalização para a faixa `[0, 1]` ajuda na estabilidade numérica, melhora a escala dos gradientes e costuma acelerar a convergência.

In [4]:
from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split

def load_data(seed, test_size=0.2):
    (X_train, y_train), _ = cifar10.load_data()

    X_train = X_train.reshape(X_train.shape[0], -1).astype("float32") / 255.0
    y_train = y_train.ravel()

    X_train, X_val, y_train, y_val = train_test_split(
        X_train,
        y_train,
        test_size=test_size,
        random_state=seed,
        stratify=y_train,
    )

    return X_train, X_val, y_train, y_val

test = load_data(seed=42)
print(test[0].shape, test[1].shape, test[2].shape, test[3].shape)

(40000, 3072) (10000, 3072) (40000,) (10000,)


# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [5]:
from sklearn.neural_network import MLPClassifier

def train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=200,
    )
    model.fit(X_train, y_train.ravel() if len(y_train.shape) > 1 else y_train)
    return model

test_model = train_mlp(
    X_train=test[0],
    y_train=test[2],
    activation="relu",
    hidden_layers=(100,),
    learning_rate=0.001,
    seed=42
)
print(test_model)

MLPClassifier(random_state=42)


### Respostas Q2:

1. Na primeira camada, o número de parâmetros é dado por
número de entradas × número de neurônios + número de vieses.
Para CIFAR-10 após flatten, cada imagem tem 3072 entradas.

2. ReLU aplica a função 
f(x) = max(0,x). Ela introduz não linearidade, costuma acelerar o treinamento e ajuda a reduzir problemas de gradiente muito pequeno.

3. MLPs têm muitos parâmetros porque conectam cada pixel de entrada a todos os neurônios da camada seguinte. Como imagens têm alta dimensionalidade, o número de pesos cresce muito rápido.

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [6]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, X_test, y_test):
    y_true = y_test.ravel() if len(y_test.shape) > 1 else y_test
    y_pred = model.predict(X_test)

    results = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_score": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

    return pd.DataFrame([results])

test_results = evaluate(test_model, test[1], test[3])
print(test_results)

   accuracy  precision  recall  f1_score
0      0.44   0.435051    0.44  0.431925


### Resposta Q3:

1. Accuracy representa a proporção de acertos em relação ao total de previsões. Em termos simples, mede quantas amostras foram classificadas corretamente.

2. Precision mede, entre as previsões positivas feitas pelo modelo, quantas estavam corretas. Recall mede, entre os casos realmente positivos, quantos o modelo conseguiu শন? melhor evitar; "conseguiu identificar". Precision foca em falsos positivos; recall foca em falsos negativos.

3. O f1-score é importante quando você quer equilibrar precision e recall, especialmente em problemas com classes desbalanceadas ou quando errar por excesso de falsos positivos ou falsos negativos tem custo relevante.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
- hidden_layers=(256,128) com learning_rate=0.001, melhor val_accuracy ≈ 0.5087 (f1_score ≈ 0.5026). Bom trade‑off de acurácia apesar do maior custo de treino e alguma tendência a overfitting.
2. Qual configuração apresentou maior estabilidade?
- learning_rate=0.001 (estabilidade geral entre runs); entre ativações, logistic/tanh mostraram maior estabilidade da validação que relu nas mesmas condições.
3. Qual o benefício do rastreamento experimental?
- permite comparar runs sistematicamente (parâmetros × métricas × tempo), reproduzir experimentos, identificar rapidamente melhores hiperparâmetros, salvar artefatos/metadados e facilitar análise/colaboração.

**Solução**:

In [7]:
import time
import mlflow

def run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    activation,
    hidden_layers,
    learning_rate,
    seed,
    max_iter=200,
    batch_size=32,
):
    with mlflow.start_run():
        start_time = time.time()

        model = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            learning_rate_init=learning_rate,
            random_state=seed,
            max_iter=max_iter,
            batch_size=batch_size,
        )
        model.fit(X_train, y_train.ravel() if len(y_train.shape) > 1 else y_train)

        training_time = time.time() - start_time
        metrics_df = evaluate(model, X_val, y_val)
        metrics = metrics_df.iloc[0].to_dict()

        mlflow.log_param("activation", activation)
        mlflow.log_param("hidden_layers", hidden_layers)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("batch_size", batch_size)

        mlflow.log_metric("accuracy", float(metrics["accuracy"]))
        mlflow.log_metric("precision", float(metrics["precision"]))
        mlflow.log_metric("recall", float(metrics["recall"]))
        mlflow.log_metric("f1_score", float(metrics["f1_score"]))
        mlflow.log_metric("training_time", float(training_time))

    return model, metrics_df

In [8]:
data = load_data(seed=42)
X_train, X_val, y_train, y_val = data

model, results = run_experiment(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    activation="relu",
    hidden_layers=(100,),
    learning_rate=0.01,
    seed=42,
    max_iter=10,
    batch_size=2048,
)
results

,accuracy,precision,recall,f1_score
0,0.186,0.09949,0.186,0.11768


# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
- tanh (menor final_loss = 1.233027). Observação: todas as execuções atingiram n_iter=100 (max_iter) — nenhuma convergiu antes de max_iter.
2. Qual ativação apresentou maior estabilidade?
-  logistic mostrou melhor desempenho de validação (val_accuracy=0.5025) mantendo train_accuracy próximo (0.5728), indicando estabilidade relativa entre treino e validação. relu teve a menor lacuna treino–val (menos overfitting) porém desempenho geral ruim
3. Houve diferenças significativas no treinamento?
- Sim. relu performou pior (maior final_loss=1.542440 e menor val_accuracy=0.4385). logistic e tanh são pareados (perfis próximos), com logistic ligeiramente melhor em validação e tanh com perda final um pouco menor.
4. Por que a ReLU é amplamente utilizada em Deep Learning?
- Porque é computacionalmente simples, produz ativações esparsas e ajuda a mitigar o problema do gradiente desaparecendo em redes profundas, acelerando convergência em arquiteturas profundas. Em redes rasas/MLPs e com configuração/hyperparâmetros específicos, ReLU pode às vezes exigir ajuste (learning rate, inicialização, regularização) e portanto ficar atrás de outras ativações.

**Solução**:

In [9]:
import time

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


def _prepare_targets(y):
    return y.ravel() if len(y.shape) > 1 else y


def _run_mlp_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    activation,
    hidden_layers,
    learning_rate,
    seed,
    max_iter=20,
    batch_size=32,
    run_name=None,
):
    y_train_1d = _prepare_targets(y_train)
    y_val_1d = _prepare_targets(y_val)

    with mlflow.start_run(run_name=run_name):
        start_time = time.time()

        model = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            learning_rate_init=learning_rate,
            random_state=seed,
            max_iter=max_iter,
            batch_size=batch_size,
        )
        model.fit(X_train, y_train_1d)

        training_time = time.time() - start_time
        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)

        results = {
            "activation": activation,
            "hidden_layers": hidden_layers,
            "learning_rate": learning_rate,
            "max_iter": max_iter,
            "batch_size": batch_size,
            "training_time": training_time,
            "n_iter": getattr(model, "n_iter_", None),
            "final_loss": float(model.loss_),
            "train_accuracy": accuracy_score(y_train_1d, y_train_pred),
            "val_accuracy": accuracy_score(y_val_1d, y_val_pred),
            "precision": precision_score(y_val_1d, y_val_pred, average="macro", zero_division=0),
            "recall": recall_score(y_val_1d, y_val_pred, average="macro", zero_division=0),
            "f1_score": f1_score(y_val_1d, y_val_pred, average="macro", zero_division=0),
        }

        mlflow.log_param("activation", activation)
        mlflow.log_param("hidden_layers", hidden_layers)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("batch_size", batch_size)

        mlflow.log_metric("accuracy", results["val_accuracy"])
        mlflow.log_metric("precision", results["precision"])
        mlflow.log_metric("recall", results["recall"])
        mlflow.log_metric("f1_score", results["f1_score"])
        mlflow.log_metric("training_time", training_time)
        mlflow.log_metric("final_loss", results["final_loss"])
        mlflow.log_metric("train_accuracy", results["train_accuracy"])
        mlflow.log_metric("n_iter", float(results["n_iter"] or 0))

    return model, pd.DataFrame([results])


activations_results = []
for activation in ["logistic", "tanh", "relu"]:
    _, metrics_df = _run_mlp_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation=activation,
        hidden_layers=(100,),
        learning_rate=0.001,
        seed=42,
        max_iter=100,
        batch_size=2048,
        run_name=f"q5_{activation}",
    )
    activations_results.append(metrics_df)

q5_results = pd.concat(activations_results, ignore_index=True).sort_values(
    by=["val_accuracy", "f1_score"], ascending=False
)
q5_results

,activation,hidden_layers,learning_rate,max_iter,batch_size,training_time,n_iter,final_loss,train_accuracy,val_accuracy,precision,recall,f1_score
0,logistic,"(100,)",0.001,100,2048,66.046521,100,1.244828,0.572775,0.5025,0.507017,0.5025,0.501035
1,tanh,"(100,)",0.001,100,2048,51.257254,100,1.233027,0.573625,0.4877,0.490991,0.4877,0.483608
2,relu,"(100,)",0.001,100,2048,50.262715,100,1.542440,0.464750,0.4385,0.436585,0.4385,0.432648


# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
- Não, o desempenho aumentou ao crescer a rede até (256, 128), mas pequenas redes tiveram desempenho bem inferior.
2. Qual arquitetura apresentou melhor tradeoff?
- (256, 128), maior val_accuracy = 0.5087 e f1_score = 0.5026, porém com custo maior; se priorizar custo/performance, (128, 64) é um compromisso razoável (val_accuracy=0.4923, tempo=61.84s).
3. Houve sinais de overfitting?
- Sim, lacuna entre treino e validação aumenta com o tamanho:
(256,128) gap = 0.6125 − 0.5087 = 0.1038
(128,64) gap = 0.5486 − 0.4923 = 0.0563
(64,) gap = 0.4367 − 0.4140 = 0.0227
(32,) gap = 0.3579 − 0.3486 = 0.0093
4. Qual arquitetura apresentou maior custo computacional?
- (256, 128), training_time ≈ 99.54 s (maior custo de treino entre os testados).

**Solução**:

In [10]:
architectures_results = []
for hidden_layers in [(32,), (64,), (128, 64), (256, 128)]:
    _, metrics_df = _run_mlp_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation="relu",
        hidden_layers=hidden_layers,
        learning_rate=0.001,
        seed=42,
        max_iter=100,
        batch_size=2048,
        run_name=f"q6_{hidden_layers}",
    )
    architectures_results.append(metrics_df)

q6_results = pd.concat(architectures_results, ignore_index=True).sort_values(
    by=["val_accuracy", "f1_score"], ascending=False
)
q6_results

,activation,hidden_layers,learning_rate,max_iter,batch_size,training_time,n_iter,final_loss,train_accuracy,val_accuracy,precision,recall,f1_score
3,relu,"(256, 128)",0.001,100,2048,99.535002,100,1.104046,0.612525,0.5087,0.516070,0.5087,0.502625
2,relu,"(128, 64)",0.001,100,2048,61.843088,100,1.299959,0.548625,0.4923,0.491290,0.4923,0.487704
1,relu,"(64,)",0.001,100,2048,37.625280,100,1.605560,0.436700,0.4140,0.410650,0.4140,0.409862
0,relu,"(32,)",0.001,100,2048,31.734820,100,1.779484,0.357925,0.3486,0.344605,0.3486,0.338132


# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
- 0.001, val_accuracy=0.4385, f1_score=0.4326.
2. Qual apresentou maior instabilidade?
- 0.1, treino falhou/divergiu (treino/val accuracy=0.10, n_iter=19, perda alta), indicando comportamento instável; 0.01 também mostrou comportamento ruim (val_accuracy=0.1985, final_loss=2.0346).
3. O que acontece quando o learning rate é muito alto?
- treinamento diverge ou oscila, perda aumenta e o modelo pode colapsar em previsões triviais (ex.: accuracy ≈ 1/10), como em 0.1.
4. O que acontece quando o learning rate é muito baixo?
- aprendizagem muito lenta; converge lentamente ou não atinge bons mínimos dentro do max_iter (mas é estável), em 0.001 obteve o melhor resultado aqui mesmo tendo n_iter=100 (atingiu max_iter).

In [11]:
learning_rate_results = []
for learning_rate in [0.1, 0.01, 0.001]:
    _, metrics_df = _run_mlp_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation="relu",
        hidden_layers=(100,),
        learning_rate=learning_rate,
        seed=42,
        max_iter=100,
        batch_size=2048,
        run_name=f"q7_lr_{learning_rate}",
    )
    learning_rate_results.append(metrics_df)

q7_results = pd.concat(learning_rate_results, ignore_index=True).sort_values(
    by=["val_accuracy", "f1_score"], ascending=False
)
q7_results

,activation,hidden_layers,learning_rate,max_iter,batch_size,training_time,n_iter,final_loss,train_accuracy,val_accuracy,precision,recall,f1_score
2,relu,"(100,)",0.001,100,2048,50.258807,100,1.542440,0.464750,0.4385,0.436585,0.4385,0.432648
1,relu,"(100,)",0.010,100,2048,49.068714,98,2.034588,0.196675,0.1985,0.113917,0.1985,0.120208
0,relu,"(100,)",0.100,100,2048,9.773460,19,2.305674,0.100000,0.1000,0.010000,0.1000,0.018182


# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
- hidden_layers=(256,128) (melhor val_accuracy ≈ 0.5087) com learning rate 0.001.
2. Quais foram as principais dificuldades observadas?
- instabilidade com lr alto, treinamento lento para redes grandes, overfitting em redes maiores, e sensibilidade de relu a hiperparâmetros.
3. Por que MLPs possuem limitações para imagens?
- não exploram estrutura espacial, geram enorme número de parâmetros e são computacionalmente ineficientes para imagens.
4. Como o backpropagation contribui para o aprendizado da rede?
- permite atualizar pesos via gradientes do erro; controla como a rede aprende — necessita de bom learning_rate, inicialização e regularização para convergir de forma estável.

## Respostas Q8

#### *Loss Behavior:*
 final_loss menor em redes maiores e em tanh (melhor mínimo observado); learning rates altos (0.01/0.1) fizeram a loss aumentar/divergir.
#### *Impacto do Learning Rate:*
 0.001 foi estável e obteve melhores métricas; 0.01 e 0.1 causaram instabilidade/divergência (treino precoce ou collapse).

#### *Impacto da Arquitetura:*
 aumentar a rede melhorou val_accuracy até  (256, 128) mas incrementou overfitting e tempo de treino (gap treino/val e training_time aumentaram).
#### *Impacto das Ativações:*
 logistic e tanh performaram melhor aqui (val_accuracy ~0.50), tanh teve menor loss; relu ficou pior possivelmente por ajuste de hiperparâmetros (lr/initialization) e pelo fato de MLP ser rasa.
#### *Behavior of Training:*
 a maioria dos runs atingiu max_iter (não convergiram antes), maiores redes e batchs grandes aumentaram tempo; lr muito alto reduziu n_iter por parada prematura com má performance.
#### *Limitations of MLP for Images:*
 alta dimensionalidade → muitos pesos; nenhuma injeção de viés espacial; custo computacional alto e tendência a overfit — arquiteturas convolucionais são mais adequadas.
#### *Backpropagation & Learning:*
 backprop calcula gradientes e atualiza pesos; seu desempenho depende de learning rate, inicialização, regularização e arquitetura — controla velocidade/estabilidade da aprendizagem.